In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option("display.max_columns", None)

In [3]:
df = pd.read_csv('../raw_data/investments_VC.csv', encoding='latin1', low_memory=False)
print(df.shape)

(54294, 39)


In [4]:
# drops all rows with more than 50% missing data
row_missing_pct = df.isna().mean(axis=1).mul(100)
df = df[row_missing_pct <= 50]

print(f"New shape: {df.shape}")

New shape: (49438, 39)


In [5]:
# printing missing values
missing = pd.DataFrame({
'missing_count': df.isna().sum(),
'missing_pct': df.isna().mean().mul(100).round(2),
'dtype': df.dtypes
}).sort_values('missing_pct', ascending=False)

print(missing)

                      missing_count  missing_pct    dtype
state_code                    19277        38.99   object
founded_year                  10956        22.16  float64
founded_quarter               10956        22.16   object
founded_month                 10956        22.16   object
founded_at                    10884        22.02   object
city                           6116        12.37   object
country_code                   5273        10.67   object
region                         5273        10.67   object
 market                        3968         8.03   object
category_list                  3961         8.01   object
homepage_url                   3449         6.98   object
status                         1314         2.66   object
round_C                           0         0.00  float64
post_ipo_debt                     0         0.00  float64
secondary_market                  0         0.00  float64
product_crowdfunding              0         0.00  float64
round_A       

In [6]:
cols_to_drop = [
'city',
'founded_quarter', # can be derived from founded_at
'founded_month', # can be derived from founded_at
'founded_year', # can be derived from founded_at
'homepage_url',
'name',
]

df = df.drop(columns=cols_to_drop)
print(f"New shape: {df.shape}")

New shape: (49438, 33)


In [7]:
# Check current types
print(df.dtypes)

permalink                object
category_list            object
 market                  object
 funding_total_usd       object
status                   object
country_code             object
state_code               object
region                   object
funding_rounds          float64
founded_at               object
first_funding_at         object
last_funding_at          object
seed                    float64
venture                 float64
equity_crowdfunding     float64
undisclosed             float64
convertible_note        float64
debt_financing          float64
angel                   float64
grant                   float64
private_equity          float64
post_ipo_equity         float64
post_ipo_debt           float64
secondary_market        float64
product_crowdfunding    float64
round_A                 float64
round_B                 float64
round_C                 float64
round_D                 float64
round_E                 float64
round_F                 float64
round_G 

In [8]:
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col] = df[col].str.strip()   # remove whitespace
    df[col] = df[col].str.lower()   # standardise case

print("String columns cleaned")

String columns cleaned


In [9]:
# This fixes the column NAMES/HEADERS
df.columns = df.columns.str.strip()

In [10]:
print(df.dtypes)

permalink                object
category_list            object
market                   object
funding_total_usd        object
status                   object
country_code             object
state_code               object
region                   object
funding_rounds          float64
founded_at               object
first_funding_at         object
last_funding_at          object
seed                    float64
venture                 float64
equity_crowdfunding     float64
undisclosed             float64
convertible_note        float64
debt_financing          float64
angel                   float64
grant                   float64
private_equity          float64
post_ipo_equity         float64
post_ipo_debt           float64
secondary_market        float64
product_crowdfunding    float64
round_A                 float64
round_B                 float64
round_C                 float64
round_D                 float64
round_E                 float64
round_F                 float64
round_G 

In [11]:
# Convert all date columns from object to datetime
date_cols = ['founded_at', 'first_funding_at', 'last_funding_at']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Clean funding_total_usd before converting to numeric
df['funding_total_usd'] = (df['funding_total_usd']
.str.replace(',', '', regex=False) # remove US style commas
.str.replace('-', '0', regex=False) # convert dashes to 0
.str.replace('$', '', regex=False) # remove currency symbols
)

df['funding_total_usd'] = pd.to_numeric(df['funding_total_usd'], errors='coerce')


# Verify the changes
print("\nAfter conversion:")
print(df[date_cols + ['funding_total_usd']].dtypes)
print(f"NaNs after conversion: {df['funding_total_usd'].isna().sum()}")
print(df['funding_total_usd'].describe())


After conversion:
founded_at           datetime64[ns]
first_funding_at     datetime64[ns]
last_funding_at      datetime64[ns]
funding_total_usd             int64
dtype: object
NaNs after conversion: 0
count    4.943800e+04
mean     1.316667e+07
std      1.535540e+08
min      0.000000e+00
25%      5.000000e+04
50%      1.000000e+06
75%      6.772162e+06
max      3.007950e+10
Name: funding_total_usd, dtype: float64


In [12]:
print(df.shape)

(49438, 33)


In [13]:
# How many companies founded before 2000
print(f"Companies founded before 2000: {(df['founded_at'] < '2000-01-01').sum()}")

Companies founded before 2000: 3730


In [14]:
df = df[(df['founded_at'] >= '2000-01-01') | (df['founded_at'].isna())]
print(f"New shape: {df.shape}")

New shape: (45708, 33)


In [15]:
# Checking for missing values again after updates to df were made
missing = pd.DataFrame({
'missing_count': df.isna().sum(),
'missing_pct': df.isna().mean().mul(100).round(2),
'dtype': df.dtypes
}).sort_values('missing_pct', ascending=False)

print(missing)

                      missing_count  missing_pct           dtype
state_code                    18353        40.15          object
founded_at                    10885        23.81  datetime64[ns]
country_code                   5128        11.22          object
region                         5128        11.22          object
market                         3645         7.97          object
category_list                  3638         7.96          object
status                         1173         2.57          object
first_funding_at                  9         0.02  datetime64[ns]
last_funding_at                   6         0.01  datetime64[ns]
round_C                           0         0.00         float64
secondary_market                  0         0.00         float64
product_crowdfunding              0         0.00         float64
round_A                           0         0.00         float64
round_B                           0         0.00         float64
permalink                

In [16]:
df.head(10)

,permalink,category_list,market,funding_total_usd,status,country_code,state_code,region,funding_rounds,founded_at,first_funding_at,last_funding_at,seed,venture,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,private_equity,post_ipo_equity,post_ipo_debt,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,/organization/waywire,|entertainment|politics|social media|news|,news,1750000,acquired,usa,ny,new york city,1.0,2012-06-01,2012-06-30,2012-06-30,1750000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/organization/tv-communications,|games|,games,4000000,operating,usa,ca,los angeles,2.0,NaT,2010-06-04,2010-09-23,0.0,4000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/organization/rock-your-paper,|publishing|education|,publishing,40000,operating,est,NaN,tallinn,1.0,2012-10-26,2012-08-09,2012-08-09,40000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,/organization/in-touch-network,|electronics|guides|coffee|restaurants|music|i...,electronics,1500000,operating,gbr,NaN,london,1.0,2011-04-01,2011-04-01,2011-04-01,1500000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,/organization/r-ranch-and-mine,|tourism|entertainment|games|,tourism,60000,operating,usa,tx,dallas,2.0,2014-01-01,2014-08-17,2014-09-26,0.0,0.0,60000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,/organization/club-domains,|software|,software,7000000,NaN,usa,fl,ft. lauderdale,1.0,2011-10-10,2013-05-31,2013-05-31,0.0,7000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7000000.0,0.0,0.0,0.0,0.0,0.0,0.0
6,/organization/fox-networks,|advertising|,advertising,4912393,closed,arg,NaN,buenos aires,1.0,NaT,2007-01-16,2007-01-16,0.0,0.0,0.0,4912393.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,/organization/0-6-com,|curated web|,curated web,2000000,operating,NaN,NaN,NaN,1.0,2007-01-01,2008-03-19,2008-03-19,0.0,2000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,/organization/004-technologies,|software|,software,0,operating,usa,il,"springfield, illinois",1.0,2010-01-01,2014-07-24,2014-07-24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,/organization/01games-technology,|games|,games,41250,operating,hkg,NaN,hong kong,1.0,NaT,2014-07-01,2014-07-01,41250.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
# Categorical columns - fill with 'unknown'
cat_cols = ['state_code', 'country_code', 'region', 'market', 'category_list', 'status']
df[cat_cols] = df[cat_cols].fillna('unknown')


# Dates - fill with median
df['founded_at'] = df['founded_at'].fillna(df['founded_at'].median())
df['first_funding_at'] = df['first_funding_at'].fillna(df['first_funding_at'].median())
df['last_funding_at'] = df['last_funding_at'].fillna(df['last_funding_at'].median())

# Verify nothing is missing
print(df.isna().sum().sort_values(ascending=False))

permalink               0
debt_financing          0
round_G                 0
round_F                 0
round_E                 0
round_D                 0
round_C                 0
round_B                 0
round_A                 0
product_crowdfunding    0
secondary_market        0
post_ipo_debt           0
post_ipo_equity         0
private_equity          0
grant                   0
angel                   0
convertible_note        0
category_list           0
undisclosed             0
equity_crowdfunding     0
venture                 0
seed                    0
last_funding_at         0
first_funding_at        0
founded_at              0
funding_rounds          0
region                  0
state_code              0
country_code            0
status                  0
funding_total_usd       0
market                  0
round_H                 0
dtype: int64
